# VN-ALPR — Train trên Google Colab

Runtime → Change runtime type → **T4 GPU**. Chạy lần lượt từng ô.

1. Cài đặt  2. Dữ liệu  3. Train detector  4. Train CRNN  5. Đánh giá + ablation  6. Export + benchmark  7. Tải model về

In [3]:
# 1. Cài đặt
!git clone https://github.com/hungdt31/vn-alpr.git
%cd vn-alpr
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into 'vn-alpr'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 89 (delta 4), reused 83 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 61.57 KiB | 1000.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/vn-alpr
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 10.3 MB/s eta 0:00:00
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Dữ liệu

Cách A: tải dataset YOLO từ Roboflow Universe (đặt API key trong Colab Secrets tên `ROBOFLOW_API_KEY`).
Cách B: upload `data/raw` (images/, labels/, texts.csv, test_texts.csv) lên Google Drive rồi copy vào.

In [4]:
# Cách A — Roboflow (điền workspace/project/version của dataset bạn chọn)
!pip install -q roboflow
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace('school-fuhih').project('vietnamese-license-plate-tptd0').version(1).download('yolov8', location='data/roboflow')

# gộp train/valid/test của Roboflow vào data/raw để tự chia lại theo seed cố định
import shutil, pathlib
for split in ['train', 'valid', 'test']:
    for kind in ['images', 'labels']:
        src = pathlib.Path('data/roboflow') / split / kind
        dst = pathlib.Path('data/raw') / kind
        dst.mkdir(parents=True, exist_ok=True)
        if src.exists():
            for f in src.iterdir():
                shutil.copy2(f, dst / f.name)
print(len(list(pathlib.Path('data/raw/images').iterdir())), 'images')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/roboflow in yolov8:: 100%|██████████| 16726/16726 [00:01<00:00, 8528.30it/s] 


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
8357 images


In [6]:
# Cách B — từ Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/vn-alpr-data/raw data/

# Chia train/val/test (ảnh tự chụp own_* luôn vào test)
!python scripts/split_dataset.py --src data/raw --dst data/processed/detection --test-prefix own_

test : 1254 images
val  : 1254 images
train: 5849 images


In [8]:
# 3. Train detector (~30-60 phút trên T4 tuỳ số ảnh)
!python scripts/train_detector.py --model yolo11n.pt --epochs 20 --batch 32
!python scripts/train_detector.py --eval models/detector/best.pt

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/data_plate.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate-3, nbs=64, nms=None, opset=None, optimize=False, optimizer=

In [9]:
# 4a. Pre-train CRNN trên dữ liệu tổng hợp
!python scripts/gen_synthetic.py --n 50000 --out data/synthetic/ocr --seed 0
!python scripts/gen_synthetic.py --n 2000 --out data/synthetic/ocr_val --seed 1
!python scripts/train_crnn.py --config configs/crnn.yaml --epochs 20 --train-dirs data/synthetic/ocr --val-dirs data/synthetic/ocr_val

5000/50000
10000/50000
15000/50000
20000/50000
25000/50000
30000/50000
35000/50000
40000/50000
45000/50000
50000/50000
wrote 75013 line images -> data/synthetic/ocr
wrote 2976 line images -> data/synthetic/ocr_val
train 75013 lines, val 2976 lines, device cuda
epoch   1  loss 4.6921  143s  val line acc 0.0000  char acc 0.0000  *saved*
epoch   2  loss 0.6096  118s  val line acc 0.9923  char acc 0.9985  *saved*
epoch   3  loss 0.0203  116s  val line acc 0.9966  char acc 0.9994  *saved*
epoch   4  loss 0.0082  118s  val line acc 0.9872  char acc 0.9977
epoch   5  loss 0.0033  114s  val line acc 0.9993  char acc 0.9999  *saved*
epoch   6  loss 0.0123  116s  val line acc 0.9987  char acc 0.9998
epoch   7  loss 0.0025  114s  val line acc 1.0000  char acc 1.0000  *saved*
epoch   8  loss 0.0018  113s  val line acc 0.6267  char acc 0.8975
epoch   9  loss 0.0020  112s  val line acc 1.0000  char acc 1.0000
epoch  10  loss 0.0010  113s  val line acc 0.9987  char acc 0.9998
epoch  11  loss 0.0015  

In [10]:
# 4b. Fine-tune CRNN trên crop biển số thật (cần data/raw/texts.csv)
!python scripts/make_ocr_crops.py --detection data/processed/detection --texts data/raw/texts.csv --out data/processed/ocr
!python scripts/train_crnn.py --config configs/crnn.yaml --resume models/crnn/best.pt --epochs 30 --lr 0.0005
!python scripts/train_crnn.py --config configs/crnn.yaml --eval models/crnn/best.pt

Traceback (most recent call last):
  File "/content/vn-alpr/scripts/make_ocr_crops.py", line 84, in <module>
    main()
    ~~~~^^
  File "/content/vn-alpr/scripts/make_ocr_crops.py", line 49, in main
    with open(args.texts, encoding="utf-8") as f:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/texts.csv'
train 75013 lines, val 0 lines, device cuda
epoch   1  loss 0.0004  116s
epoch   2  loss 0.0003  113s
epoch   3  loss 0.0003  112s
epoch   4  loss 0.0008  114s
epoch   5  loss 0.0007  117s
epoch   6  loss 0.0010  118s
epoch   7  loss 0.0011  115s
epoch   8  loss 0.0014  116s
epoch   9  loss 0.0010  115s
epoch  10  loss 0.0005  116s
epoch  11  loss 0.0026  115s
epoch  12  loss 0.0003  117s
epoch  13  loss 0.0004  118s
epoch  14  loss 0.0003  122s
epoch  15  loss 0.0002  121s
epoch  16  loss 0.0006  121s
epoch  17  loss 0.0003  119s
epoch  18  loss 0.0001  116s
epoch  19  loss 0.0001  120s
epoch  20  loss 0.0001  115s
epoc

In [11]:
# 5. Đánh giá end-to-end + ablation (kết quả cộng dồn vào outputs/ablation.csv)
TEST = '--images data/processed/detection/images/test --gt data/raw/test_texts.csv'
!pip install -q -r requirements-paddle.txt
!python scripts/evaluate.py {TEST} --ocr paddle --no-postprocess --tag "YOLO + PaddleOCR"
!python scripts/evaluate.py {TEST} --ocr paddle --tag "YOLO + PaddleOCR + postprocess"
!python scripts/evaluate.py {TEST} --no-postprocess --tag "YOLO + CRNN"
!python scripts/evaluate.py {TEST} --tag "YOLO + CRNN + postprocess"
import pandas as pd; pd.read_csv('outputs/ablation.csv')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/ablation.csv'

In [12]:
# 6. Export + benchmark (cần 1 video demo, ví dụ data/demo.mp4)
!python scripts/export_models.py --detector models/detector/best.pt --crnn models/crnn/best.pt
!python scripts/benchmark.py --source data/demo.mp4 --n 300
!python scripts/benchmark.py --source data/demo.mp4 --n 300 --detector models/detector/best.onnx --crnn models/crnn/best.onnx
# TensorRT FP16:
!python scripts/export_models.py --detector models/detector/best.pt --format engine --half
!python scripts/benchmark.py --source data/demo.mp4 --n 300 --detector models/detector/best.engine --crnn models/crnn/best.onnx

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from 'models/detector/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 344ms
Prepared 2 packages in 34ms
Installed 2 packages in 8ms
 + colorama==0.4.6
 + onnxslim==0.1.97

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxsl

In [13]:
# 7. Tải model + kết quả về máy
!zip -r vn_alpr_models.zip models outputs runs/detector/*/results.png runs/detector/*/confusion_matrix.png
from google.colab import files; files.download('vn_alpr_models.zip')

	zip warning: name not matched: outputs
	zip warning: name not matched: runs/detector/*/results.png
	zip warning: name not matched: runs/detector/*/confusion_matrix.png
  adding: models/ (stored 0%)
  adding: models/README.md (deflated 39%)
  adding: models/crnn/ (stored 0%)
  adding: models/crnn/best.json (deflated 17%)
  adding: models/crnn/best.onnx (deflated 8%)
  adding: models/crnn/best.pt (deflated 8%)
  adding: models/detector/ (stored 0%)
  adding: models/detector/best.onnx (deflated 13%)
  adding: models/detector/best.fp16.onnx (deflated 11%)
  adding: models/detector/best.engine (deflated 89%)
  adding: models/detector/best.pt (deflated 10%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>